In [402]:
import pandas as pd 
import numpy as np


import matplotlib.pyplot as plt

In [403]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [404]:
#!wget $data -O data-week-3.csv

In [405]:
df = pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [406]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [407]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [408]:
df.head().T

,0,1,2,3,4
customerid,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
seniorcitizen,0,0,0,0,0
partner,Yes,No,No,No,No
dependents,No,No,No,No,No
tenure,1,34,2,45,2
phoneservice,No,Yes,Yes,No,Yes
multiplelines,No phone service,No,No,No phone service,No
internetservice,DSL,DSL,DSL,DSL,Fiber optic
onlinesecurity,No,Yes,Yes,Yes,No


In [409]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [410]:
tc = pd.to_numeric(df.totalcharges, errors='coerce')
tc

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: totalcharges, Length: 7043, dtype: float64

In [411]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

In [412]:
df.churn.head()

0     No
1     No
2    Yes
3     No
4    Yes
Name: churn, dtype: str

In [413]:
df.churn = (df.churn == 'Yes').astype(int)

In [414]:
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

# 3.3 Setting up the validation framework

Perform the train/validation/test split with Scikit-Learn

In [415]:
from sklearn.model_selection import train_test_split


In [416]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [417]:
len(df_full_train), len(df_test)

(5634, 1409)

In [418]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [419]:
len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

In [420]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [421]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [422]:
df_train.head().T

,0,1,2,3,4
customerid,8015-IHCGW,1960-UYCNN,9250-WYPLL,6786-OBWQR,1328-EUZHC
gender,Female,Male,Female,Female,Female
seniorcitizen,0,0,0,0,0
partner,Yes,No,No,Yes,Yes
dependents,Yes,No,No,Yes,No
tenure,72,10,5,5,18
phoneservice,Yes,Yes,Yes,Yes,Yes
multiplelines,Yes,Yes,Yes,No,No
internetservice,Fiber optic,Fiber optic,Fiber optic,Fiber optic,No
onlinesecurity,Yes,No,No,No,No internet service


In [423]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

In [424]:
df_train.head().T

,0,1,2,3,4
customerid,8015-IHCGW,1960-UYCNN,9250-WYPLL,6786-OBWQR,1328-EUZHC
gender,Female,Male,Female,Female,Female
seniorcitizen,0,0,0,0,0
partner,Yes,No,No,Yes,Yes
dependents,Yes,No,No,Yes,No
tenure,72,10,5,5,18
phoneservice,Yes,Yes,Yes,Yes,Yes
multiplelines,Yes,Yes,Yes,No,No
internetservice,Fiber optic,Fiber optic,Fiber optic,Fiber optic,No
onlinesecurity,Yes,No,No,No,No internet service


In [425]:
df_full_train = df_full_train.reset_index(drop=True)

In [426]:
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [427]:
df_full_train.churn.value_counts()

churn
0    4113
1    1521
Name: count, dtype: int64

In [428]:
df_full_train.churn.value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [429]:
global_churn_rate = df_full_train.churn.mean()
round(global_churn_rate,2)

np.float64(0.27)

In [430]:
df_full_train.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

In [431]:
numerical =['tenure', 'monthlycharges', 'totalcharges']

In [432]:
df_full_train.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='str')

In [433]:
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
        'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [434]:
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

3.5. Feature importance: Churn rate and risk ratio

Churn rate

In [435]:
df_full_train.head().T

,0,1,2,3,4
customerid,5442-PPTJY,6261-RCVNS,2176-OSJUV,6161-ERDGD,2364-UFROM
gender,Male,Female,Male,Male,Male
seniorcitizen,0,0,0,0,0
partner,Yes,No,Yes,Yes,No
dependents,Yes,No,No,Yes,No
tenure,12,42,71,71,30
phoneservice,Yes,Yes,Yes,Yes,Yes
multiplelines,No,No,Yes,Yes,No
internetservice,No,DSL,DSL,DSL,DSL
onlinesecurity,No internet service,Yes,Yes,Yes,Yes


In [436]:
churn_female = df_full_train[df_full_train.gender == 'Female'].churn.mean()
churn_female

np.float64(0.27682403433476394)

In [437]:
churn_male = df_full_train[df_full_train.gender == 'Male'].churn.mean()
churn_male

np.float64(0.2632135306553911)

In [438]:
df_full_train.partner.value_counts()

partner
No     2932
Yes    2702
Name: count, dtype: int64

In [439]:
churn_partner = df_full_train[df_full_train.partner == 'Yes'].churn.mean()

In [440]:
churn_no_partner = df_full_train[df_full_train.partner == 'No'].churn.mean()

In [441]:
global_churn = df_full_train.churn.mean()
global_churn

np.float64(0.26996805111821087)

In [442]:
global_churn - churn_partner


np.float64(0.06493474245795922)

In [443]:
global_churn - churn_no_partner

np.float64(-0.05984095297455855)

Risk ratio

In [444]:
churn_no_partner/global_churn 

np.float64(1.2216593879412643)

In [445]:
churn_partner/global_churn 

np.float64(0.7594724924338315)

In [446]:
df_group = df_full_train.groupby('gender').churn.agg(['mean', 'count'])
df_group['diff'] = df_group['mean'] - global_churn
df_group['diff_ratio'] = df_group['mean']/global_churn
df_group

,mean,count,diff,diff_ratio
gender,,,,
Female,0.276824,2796,0.006856,1.025396
Male,0.263214,2838,-0.006755,0.974980


In [447]:
from IPython.display import display

In [448]:
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['diff_ratio'] = df_group['mean']/global_churn
    display(df_group)
    print()

gender


,mean,count,diff,diff_ratio
gender,,,,
Female,0.276824,2796,0.006856,1.025396
Male,0.263214,2838,-0.006755,0.974980



seniorcitizen


,mean,count,diff,diff_ratio
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208



partner


...

,mean,count,diff,diff_ratio
contract,,,,
Month-to-month,0.431701,3104,0.161733,1.599082
One year,0.120573,1186,-0.149395,0.446621
Two year,0.028274,1344,-0.241694,0.104730



paperlessbilling


,mean,count,diff,diff_ratio
paperlessbilling,,,,
No,0.172071,2313,-0.097897,0.637375
Yes,0.338151,3321,0.068183,1.252560



paymentmethod


,mean,count,diff,diff_ratio
paymentmethod,,,,
Bank transfer (automatic),0.168171,1219,-0.101797,0.622928
Credit card (automatic),0.164339,1217,-0.105630,0.608733
Electronic check,0.455890,1893,0.185922,1.688682
Mailed check,0.193870,1305,-0.076098,0.718121


3.6 Feature importance: Mutual information

In [449]:
from sklearn.metrics import mutual_info_score

In [450]:
mutual_info_score(df_full_train.churn, df_full_train.contract)

0.0983203874041556

In [451]:
mutual_info_score(df_full_train.churn, df_full_train.gender)

0.0001174846211139946

In [452]:
mutual_info_score(df_full_train.churn, df_full_train.partner)

0.009967689095399745

In [453]:
def mutual_info_churn_score(series):
    return mutual_info_score(df_full_train.churn, series)

In [454]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

3.7 Feature importance: Correlation

In [455]:
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

In [456]:
df_full_train[df_full_train.tenure < 2].churn.mean()

np.float64(0.6247464503042597)

3.8 One hot encoding

In [457]:
from sklearn.feature_extraction import DictVectorizer


In [458]:
train_dicts = df_train[categorical+numerical].to_dict(orient='records')

In [463]:
dv = DictVectorizer(sparse=False)


In [465]:
#dv.fit(train_dicts)

In [466]:
train_dicts[0]

{'gender': 'Female',
 'seniorcitizen': 0,
 'partner': 'Yes',
 'dependents': 'Yes',
 'phoneservice': 'Yes',
 'multiplelines': 'Yes',
 'internetservice': 'Fiber optic',
 'onlinesecurity': 'Yes',
 'onlinebackup': 'Yes',
 'deviceprotection': 'Yes',
 'techsupport': 'Yes',
 'streamingtv': 'Yes',
 'streamingmovies': 'Yes',
 'contract': 'Two year',
 'paperlessbilling': 'Yes',
 'paymentmethod': 'Electronic check',
 'tenure': 72,
 'monthlycharges': 115.5,
 'totalcharges': 8425.15}

In [ ]:
X = dv.transform(train_dicts)
X

array([[0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(100, 45))